# HOMEWORK 5

In this homework you are going to implement the **Floyd-Steinberg dithering** algorithm. Dithering, in general, means that we are adding noise to the signal (in our case digital image) in order to perceive it better. In other words, by adding the noise the objective quality will be worse but the subjective quality will be better (i.e. the image will "look" better).

The details of FS dithering can be found in this [wiki](https://en.wikipedia.org/wiki/Floyd%E2%80%93Steinberg_dithering) page. In order to implement the dithering, we will implement the following steps:
* Define colour pallette
* Quantize the image to obtain the baseline and compute the average quantization error
* Implement FS dithering and compute the average quantization error

You will also have to answer the question at the end of this notebook.

Note: In this homework, you will have the chance to earn some extra points. See the "Bonus" section at the end of the notebook. Good luck!

As always, you are encouraged to use your own images :-)

In [ ]:
import cv2
import math
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

Let's load the image.

In [ ]:
# Load image
img = cv2.imread('../data/kodim23.png')
# Convert it to RGB
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
# Plot it
plt.imshow(img)

Let's start with gray tones first.

In [ ]:
# Black, dark gray, light gray, white
colors = np.array([[0, 0, 0],
                   [64, 64, 64],
                   [192, 192, 192],
                   [255, 255, 255]])


Using the colour pallette, let's quantize the original image.

In [ ]:
# Cast the image to float
img_f = img / 255.0
colors_f = colors / 255.0

# Prepare for quantization
rows, cols, channels = img_f.shape
quantized_f = np.zeros_like(img_f)

# Apply quantization
for r in range(rows):
    for c in range(cols):
        # Extract the original pixel value
        pixel = img_f[r, c]
        
        # Find the closest colour from the pallette (using absolute value/Euclidean distance)
        # Note: You may need more than one line of code here
        idx_min = np.argmin([np.linalg.norm(pixel - color) for color in colors_f])
        new_pixel = colors_f[idx_min]  
        
        # Apply quantization
        quantized_f[r, c, :] = new_pixel
        

In [ ]:
# Show quantized image (don't forget to cast back to uint8)
quantized = np.clip((quantized_f * 255), 0, 255).astype(np.uint8)
plt.imshow(quantized, cmap='gray')

In [ ]:
# Compute average quantization error
avg_quant_error = (1/(rows * cols)) * np.sum((img - quantized)**2) 
print(f"MSE: {avg_quant_error}")

# Also, PSNR
def calculate_psnr(img1, img2, max_value=255.0):
    """
    Calculates Peak Signal-to-Noise Ratio (PSNR) between two images.
    Input images should have the same dimensions and data type.
    """
    # Ensure the images are floating-point to avoid overflow/underflow errors
    img1 = img1.astype(np.float64)
    img2 = img2.astype(np.float64)
    
    # Calculate Mean Squared Error (MSE)
    mse = np.mean((img1 - img2) ** 2)
    
    # If MSE is zero, the images are identical (PSNR is infinite)
    if mse == 0:
        return float('inf')
    
    # Calculate PSNR in decibels (dB)
    psnr = 20 * np.log10(max_value / np.sqrt(mse))
    return psnr

print(f"PSNR: {calculate_psnr(img, quantized)}dB")

#### Floyd-Steinberg Dithering
We are now going to implement the FS dithering and compare it to the optimally quantized image we have calculated above.

In [ ]:
# Make a temporal copy of the original image, we will need it for error diffusion
img_tmp = np.copy(img)
dithering_f = np.zeros_like(img, dtype=np.float32)

img_tmp_f = img_tmp / 255.0

for r in range(1, rows-1):
    for c in range(1, cols-1):
        # Extract the original pixel value
        pixel = img_tmp_f[r, c]
        # Find the closest colour from the pallette (using absolute value/Euclidean distance)
        # Note: You may need more than one line of code here
        idx_min = np.argmin([np.linalg.norm(pixel - color) for color in colors_f])
        new_pixel = colors_f[idx_min]        
        
        # Compute quantization error
        quant_error = pixel - new_pixel
        # Diffuse the quantization error accroding to the FS diffusion matrix
        # Note: You may need more than one line of code here
        img_tmp_f[r,   c+1] = img_tmp_f[r,   c+1] + quant_error * 7/16
        img_tmp_f[r+1, c-1] = img_tmp_f[r+1, c-1] + quant_error * 3/16
        img_tmp_f[r+1, c  ] = img_tmp_f[r+1, c  ] + quant_error * 5/16
        img_tmp_f[r+1, c+1] = img_tmp_f[r+1, c+1] + quant_error * 1/16
        
        # Apply dithering
        dithering_f[r, c, :] = new_pixel
        
        
dithering = np.clip((dithering_f * 255), 0, 255).astype(np.uint8)

In [ ]:
# Show quantized image (don't forget to cast back to uint8)
plt.subplot(121), plt.imshow(quantized)   # optimally quantized
plt.subplot(122), plt.imshow(dithering)   # dithering

In [ ]:
# Compute average quantization error for dithered image
avg_dith_error = (1/(rows * cols)) * np.sum((img - dithering)**2) 
print(f"MSE: {avg_dith_error}")
print(f"PSNR: {calculate_psnr(img, dithering)}dB")

### Questions
* Which image has higher quantization error? Optimally quantized or dithered?

Dithered, because it enhances the subjective quality of an image, pure math is optimised in quantized images.
* Which image looks better to you?

Dithered.
* Can you repeat the same process using only two colours: black and white? Show me :-)

In [ ]:
# Wrap dithering in a function: folor + l2 for distance
def fs_dither_color(img, palette):
    img_tmp = np.copy(img)
    img_tmp_f = img_tmp / 255.0
    palette_f = palette / 255.0
    dithering_f = np.zeros_like(img, dtype=np.float32)


    for r in range(1, rows-1):
        for c in range(1, cols-1):
            # Extract the original pixel value
            pixel = img_tmp_f[r, c]
            # Find the closest colour from the pallette (using absolute value/Euclidean distance)
            # Note: You may need more than one line of code here
            idx_min = np.argmin([np.linalg.norm(pixel - color) for color in palette_f])
            new_pixel = palette_f[idx_min]        
            
            # Compute quantization error
            quant_error = pixel - new_pixel
            # Diffuse the quantization error accroding to the FS diffusion matrix
            # Note: You may need more than one line of code here
            img_tmp_f[r,   c+1] = img_tmp_f[r,   c+1] + quant_error * 7/16
            img_tmp_f[r+1, c-1] = img_tmp_f[r+1, c-1] + quant_error * 3/16
            img_tmp_f[r+1, c  ] = img_tmp_f[r+1, c  ] + quant_error * 5/16
            img_tmp_f[r+1, c+1] = img_tmp_f[r+1, c+1] + quant_error * 1/16
            
            # Apply dithering
            dithering_f[r, c, :] = new_pixel
            
            
    dithering = np.clip((dithering_f * 255), 0, 255).astype(np.uint8)
    return dithering

# FS for grayscale images + l1 for distance (just for fun)
def fs_dither_gray(img, palette):
    img_tmp_f = np.copy(img).astype(np.float32) / 255.0
    palette_f = palette.astype(np.float32) / 255.0

    rows, cols = img.shape
    dithering_f = np.zeros_like(img_tmp_f, dtype=np.float32)

    for r in range(1, rows - 1):
        for c in range(1, cols - 1):

            # Current pixel (scalar)
            pixel = img_tmp_f[r, c]

            # Find closest palette value
            idx_min = np.argmin(
                [abs(pixel - color) for color in palette_f]
            )

            # Quantized pixel
            new_pixel = palette_f[idx_min]

            # Quantization error
            quant_error = pixel - new_pixel

            # Floyd-Steinberg error diffusion
            img_tmp_f[r, c + 1] += quant_error * 7 / 16
            img_tmp_f[r + 1, c - 1] += quant_error * 3 / 16
            img_tmp_f[r + 1, c]     += quant_error * 5 / 16
            img_tmp_f[r + 1, c + 1] += quant_error * 1 / 16

            # Store quantized pixel
            dithering_f[r, c] = new_pixel

    # Convert back to 8-bit grayscale
    dithering = np.clip(
        dithering_f * 255, 0, 255
    ).astype(np.uint8)

    return dithering

In [ ]:
plt.subplot(121), plt.imshow(
    fs_dither_color(
        img=img, 
        palette=np.array(
            [[0, 0, 0],
            #  [64, 64, 64],
            #  [192, 192, 192],
            [255, 255, 255]]
        )
    )
)


plt.subplot(122), plt.imshow(
    fs_dither_gray(
        img=cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), # cv2 convertion changes the weight of each color to match human perception
        palette=np.array([0,255])
    ),
    cmap = 'gray'
)


### Bonus Points

Repeat the homework using a diffrerent image pallette. For instance, you can use an optimal colour
pallette that we can calculate via k-means algorithm. The following snippet of code will give you the 16
optimal colours for your original image.

In [ ]:
from sklearn.cluster import KMeans
for idx, n_clusters in zip([131, 132, 133],[16, 32, 256]):
    kmeans = KMeans(n_clusters=n_clusters).fit(np.reshape(img, (-1, 3)))
    colors = kmeans.cluster_centers_


    plt.subplot(idx), plt.imshow(
        fs_dither_color(
            img=img, 
            palette=colors
        )
    )

Apply FS dithering the same way you did before.
* How does the result look like to you?

At 16 there are some artifacts, the head of a read parrot looks unnatural. But on the overall, the picture looks okay.
* What happens if we use 32 colours?

The quality improves. The articats are still present (like color transit on the head on the red parrot), but less punchy.
* And what happens if we use 256 colours?

For me, the result seems indiscernible from the original.